# Walmart E-commerce Sales Pipeline

This notebook demonstrates a reproducible pandas ETL workflow for weekly retail sales. It uses the local sample export for a repeatable run and keeps the tested business logic in src/pipeline.py.

The complete DataCamp SQL result can be substituted for the local sample without changing the pipeline functions.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.pipeline import extract, transform, avg_weekly_sales_per_month, load, validation

SALES_PATH = ROOT / 'data/raw/grocery_sales_sample.csv'
EXTRA_DATA_PATH = ROOT / 'data/raw/extra_data.parquet'
CLEAN_PATH = ROOT / 'data/processed/clean_data.csv'
AGG_PATH = ROOT / 'data/processed/agg_data.csv'
REPORT_PATH = ROOT / 'reports'
REPORT_PATH.mkdir(exist_ok=True)

## 1. Extract and merge

In [ ]:
store_data = pd.read_csv(SALES_PATH, parse_dates=['Date'])
merged_df = extract(store_data, EXTRA_DATA_PATH)
print(f'Sales rows: {len(store_data):,}')
print(f'Merged rows: {len(merged_df):,}')
merged_df.head()

## 2. Transform and clean

Numeric gaps are filled with medians, the calendar month is derived from Date, and only rows with Weekly_Sales above $10,000 are retained.

In [ ]:
clean_data = transform(merged_df)
print(f'Rows after the > $10,000 filter: {len(clean_data):,}')
print(f'Remaining missing values: {int(clean_data.isna().sum().sum())}')
clean_data.head()

## 3. Aggregate monthly sales

In [ ]:
agg_data = avg_weekly_sales_per_month(clean_data)
agg_data

## 4. Load and validate

In [ ]:
load(clean_data, CLEAN_PATH, agg_data, AGG_PATH)
print({'clean_data.csv': validation(CLEAN_PATH), 'agg_data.csv': validation(AGG_PATH)})

## 5. Preliminary business analysis

In [ ]:
sns.set_theme(style='whitegrid', palette='colorblind')
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=agg_data, x='Month', y='Weekly_Sales', marker='o', color='#0F766E', ax=ax)
ax.set_title('Average weekly sales by calendar month', loc='left', weight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average weekly sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
holiday_summary = clean_data.groupby('IsHoliday', as_index=False)['Weekly_Sales'].mean()
holiday_summary['Period'] = holiday_summary['IsHoliday'].map({0: 'Regular week', 1: 'Holiday week'})
holiday_values = holiday_summary.set_index('IsHoliday')['Weekly_Sales']
holiday_lift = (holiday_values[1] / holiday_values[0] - 1) * 100
peak_month = int(agg_data.loc[agg_data['Weekly_Sales'].idxmax(), 'Month'])
print(f'Peak month in this local extract: {peak_month}')
print(f'Holiday-week average sales lift: {holiday_lift:.1f}%')
print()
print('These are descriptive results, not causal estimates. The local CSV is a sample rather than the full SQL extract.')
holiday_summary

## Interpretation and next steps

Use the full SQL extract for a production-scale result. Then compare holiday and regular weeks within the same store and department, add promotion intensity, preserve year-month ordering, and build a time-based demand forecast.